# 03 - Index Construction

Composes every parquet under `data_clean/indicators/` into a single
feature matrix keyed on `normalized_company_name`. Indicators are
discovered automatically; new indicators dropped into that folder
are picked up without changes here.

Output: `data_clean/ai_maturity_index.parquet`.

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

## 1. Compose

In [ ]:
from src.index.compose import compose_index
from src.indicators.common.io import list_indicators

available = list_indicators()
print(f"Indicators on disk: {available}")

index_df = compose_index(write=True)
print(f"Composed shape: {index_df.shape}")
index_df.head()

## 2. Optional normalization

Both z-score and min-max scalers are provided. Pick whichever the
downstream modeling step expects. Boolean / `has_*` flags are
passed through unchanged.

In [ ]:
from src.index.normalize import zscore, minmax

feature_cols = [c for c in index_df.columns if "__" in c]
z_df = zscore(index_df, columns=feature_cols)
m_df = minmax(index_df, columns=feature_cols)
z_df[feature_cols].describe().T.head()